# Data cleaning 

Cleans the story-filtered feature table produced by `src/filtered_feature_generation.py`.

**Input:** `data/features_filtered_by_ai.csv`, `data/stories_filtered.csv`  
**Output:** overwrites `cleaned_data.csv`

In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re, unicodedata
from IPython.display import clear_output
from pathlib import Path

DATA = Path("../data")
SRC = DATA / "features_filtered_by_ai.csv"
OUT = DATA / "cleaned_data.csv"
STORIES = DATA / "stories_filtered.csv"

In [9]:
# Loading in CSVs
feats = pd.read_csv(SRC, dtype={"video_id": str})
stories = pd.read_csv(STORIES, dtype={"video_id": str})

# Merging CSV by columns
feature_cols = [c for c in feats.columns if c not in ("video_id", "label", "y")]
extra_cols = ["video_id"] + [c for c in stories.columns if c not in feats.columns]
df = feats.merge(stories[extra_cols], on="video_id", how="left",
                 validate="one_to_one")

# Printing New Data Shape
print("New columns:", [c for c in extra_cols if c != "video_id"])
print("Shape:", feats.shape)
feats.info()
feats.head()

New columns: ['theme', 'url', 'title', 'source_query', 'status', 'language', 'lang_prob', 'n_words', 'transcript', 'note', 'llm_category', 'llm_confidence', 'llm_reason']
Shape: (500, 42)
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 42 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   video_id              500 non-null    str    
 1   label                 500 non-null    str    
 2   n_words_calc          500 non-null    int64  
 3   lex_var_30            500 non-null    float64
 4   mattr_30              500 non-null    float64
 5   negation_count        500 non-null    int64  
 6   negation_rate         500 non-null    float64
 7   reaction_count        500 non-null    int64  
 8   reaction_rate         500 non-null    float64
 9   contraction_count     500 non-null    int64  
 10  contraction_rate      500 non-null    float64
 11  everything_count      500 non-null    int64  
 12 

,video_id,label,n_words_calc,lex_var_30,mattr_30,negation_count,negation_rate,reaction_count,reaction_rate,contraction_count,...,profanity_rate,digit_rate,first_person_rate,second_person_rate,we_rate,repeat_bigram_rate,repeat_sentence_rate,proper_noun_rate,cta_flag,y
0,5Y4lU1gtOzo,AI,200,0.9667,0.9228,4,2.0000,0,0.0000,2,...,0.0,0.0000,0.0000,0.0000,0.0000,2.5126,0.0000,0.0000,0,1
1,TSf7LOimamk,human,110,0.9000,0.8716,5,4.5455,1,0.9091,13,...,0.0,0.0000,12.7273,3.6364,0.0000,5.5046,18.1818,7.2727,0,0
2,Exemc-4bXPM,AI,242,0.8667,0.8136,17,7.0248,0,0.0000,24,...,0.0,0.4132,5.3719,8.2645,0.8264,11.2033,2.0833,0.8264,0,1
3,wI6Mc00aspk,AI,170,0.7333,0.8556,3,1.7647,0,0.0000,2,...,0.0,1.1765,11.1765,0.5882,0.0000,4.1420,0.0000,3.5294,0,1
4,MEGD2JP5yCA,AI,56,0.9000,0.9543,1,1.7857,0,0.0000,2,...,0.0,0.0000,1.7857,3.5714,0.0000,0.0000,0.0000,7.1429,1,1


In [10]:
LOW_CONF = 0.7      # The Boundary of what is considered low confidence
N_SPOT = 10         # The number of spot checks to do for each label
REVIEW_FILE = DATA / "review_decisions.csv"

link_cols = [c for c in df.columns if re.search(r"url|link", c, re.I)]
LINK = link_cols[0] if link_cols else None     
print("Link column:", LINK)

low = df[df["llm_confidence"] < LOW_CONF]
spot = df[df["llm_confidence"] >= LOW_CONF].groupby("label").sample(n=N_SPOT // 2, random_state=0)
queue = pd.concat([low, spot]).sample(frac=1, random_state=0) 

# Reviewing the Videos
decisions = {}
if REVIEW_FILE.exists():
    decisions = pd.read_csv(REVIEW_FILE, dtype=str).set_index("video_id")["is_story"].to_dict()

todo = [v for v in queue["video_id"] if v not in decisions]
print(f"{len(queue)} to review ({len(low)} low-conf + {len(spot)} spot checks), {len(todo)} left")

for i, vid in enumerate(todo, 1):
    row = df.loc[df["video_id"] == vid].iloc[0]
    clear_output(wait=True)
    print(f"[{i}/{len(todo)}]  LLM said: {row['llm_category']} (conf {row['llm_confidence']:.2f})")
    print("LLM reason:", row["llm_reason"])
    print("Link:", row[LINK] if LINK else f"(no link column, video_id={vid})")
    print("\nTranscript:\n", str(row["transcript"])[:1200])
    ans = input("\nIs this a story? y / n / s (skip) / q (quit): ").strip().lower()
    if ans == "q":
        break
    if ans in ("y", "n"):
        decisions[vid] = ans
        (pd.Series(decisions).rename_axis("video_id").rename("is_story")
           .reset_index().to_csv(REVIEW_FILE, index=False))

# Applying the checking decisions to the dataframe
df["reviewed_story"] = df["video_id"].map(decisions)
rejected = df["reviewed_story"] == "n"
clear_output()
print(f"Reviewed {len(decisions)} | marked not-a-story: {int(rejected.sum())}")
print(df[rejected].groupby("label").size().rename("removed").to_string() if rejected.any() else "Nothing removed")
df = df[~rejected].reset_index(drop=True)
print("Rows now:", len(df))

Reviewed 8 | marked not-a-story: 2
label
AI       1
human    1
Rows now: 498


In [ ]:
# Drop free-text / LLM columns that leak the label (video_id, label and theme are kept;
# the final save in the last cell picks exactly which columns go to cleaned_data.csv)
drop_cols = [c for c in ["source_query", "title", "url", "channel",
                         "llm_category", "llm_confidence", "llm_reason"] if c in df.columns]
keep = [c for c in df.columns if c not in drop_cols or c in ("y", "n_words", "n_words_calc")]
# Always keep y and a length column if present
if "y" not in keep and "y" in df.columns:
    keep.append("y")
df = df[[c for c in df.columns if c in keep or c == "y"]].copy()

print("Columns kept:", list(df.columns))
print(df["y"].value_counts() if "y" in df.columns else "no y column")

Columns kept: ['video_id', 'n_words_calc', 'lex_var_30', 'mattr_30', 'negation_count', 'negation_rate', 'reaction_count', 'reaction_rate', 'contraction_count', 'contraction_rate', 'everything_count', 'love_count', 'well_count', 'filler_count', 'filler_rate', 'everything_rate', 'love_rate', 'well_rate', 'n_sentences', 'avg_sent_len', 'sent_len_std', 'max_sent_len', 'short_sent_rate', 'question_rate', 'exclaim_rate', 'ellipsis_rate', 'avg_word_len', 'long_word_rate', 'hapax_rate', 'ttr_full', 'top_word_share', 'profanity_rate', 'digit_rate', 'first_person_rate', 'second_person_rate', 'we_rate', 'repeat_bigram_rate', 'repeat_sentence_rate', 'proper_noun_rate', 'cta_flag', 'y', 'status', 'language', 'lang_prob', 'n_words', 'transcript', 'note', 'reviewed_story', 'transcript_clean', 'flag_odd_length']
y
1    278
0    220
Name: count, dtype: int64


In [18]:
# NLP Cleaning, normalize whitespace and unicode
def normalize(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = (text.replace("\u2018", "'").replace("\u2019", "'")
                .replace("\u201c", '"').replace("\u201d", '"'))
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)     
    return re.sub(r"\s+", " ", text).strip()

df["transcript_clean"] = df["transcript"].apply(normalize)
print("Transcripts changed:", int((df["transcript"] != df["transcript_clean"]).sum()), "of", len(df))

Transcripts changed: 0 of 498


In [20]:
# Checking for anomalies in word count and sentence length
MAX_AVG_SENT = 40  
MIN_SENTENCES = 3
DROP_FLAGGED = False  

cols = ["n_words_calc", "n_sentences", "avg_sent_len", "max_sent_len"]
print(df.groupby("label")[cols].agg(["mean", "median", "min", "max"]).round(2).T)

df["flag_odd_length"] = (df["avg_sent_len"] > MAX_AVG_SENT) | (df["n_sentences"] < MIN_SENTENCES)
print("\nFlagged by label:")
print(df.groupby("label")["flag_odd_length"].agg(flagged="sum", total="count"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["n_words_calc", "n_sentences", "avg_sent_len"]):
    sns.histplot(data=df, x=c, hue="label", bins=30, ax=ax)
plt.tight_layout(); plt.show()

display(df.loc[df["flag_odd_length"], ["video_id", "n_words_calc", "n_sentences", "avg_sent_len"]])

if DROP_FLAGGED:
    df = df[~df["flag_odd_length"]].reset_index(drop=True)
    print("Rows now:", len(df))

KeyError: 'label'

In [15]:
# Save cleaned table
assert "theme" in df.columns, "theme column missing from stories_filtered.csv"
assert "video_id" in df.columns, "video_id column missing"

# Keep video_id (needed to join transcripts later), the features, theme, and y
final = df[["video_id"] + feature_cols + ["theme", "y"]].copy()

# Checking Final Data
print("Shape:", final.shape)
print("Missing values:", int(final.isna().sum().sum()))
print(pd.crosstab(final["theme"], final["y"].map({0: "human", 1: "AI"})))
final.to_csv(OUT, index=False)
print("Saved to", OUT)

Shape: (498, 41)
Missing values: 0
y       AI  human
theme            
animal  27     27
baby    34     23
couple  35     30
family  44     35
fruit   36     14
horror  37     31
school  29     31
work    36     29
Saved to ../data/cleaned_data.csv
